# 第 6 周练习：在课程笔记本上试若干「内联」改进

本笔记本在定价任务上尝试多种传统 ML / NLP 变体，并和基线对比：

1. **XGBoost**（词袋特征上的梯度提升）
2. **专家组合（ensemble）**：融合 XGBoost、线性词袋、Word2Vec 等——两种聚合：(1) 简单平均 (2) 取两两最接近预测的平均
3. **命名实体识别（NER）**：只用实体词做词袋特征
4. **MinMax / StandardScaler**：规范化 Word2Vec 文档向量后再回归

备注：作者观察到随机森林这条线没有明显提升——读结果时对照 `Tester` 的 Error / RMSLE / Hits。


In [ ]:
# ========== 基础导入：环境、数值、可视化、序列化 ==========

# 导入 os：路径存在性判断、环境相关
import os
# 导入 math：对数误差、开方算 RMSLE
import math
# 导入 json：解析 item.details 里的 JSON 特征
import json
# 导入 random：随机基线定价器
import random
# load_dotenv：如需密钥可从 .env 加载（本格先导入常用工具）
from dotenv import load_dotenv
# HuggingFace 登录（若后续拉 Hub 数据）
from huggingface_hub import login
# matplotlib：散点图/评估可视化
import matplotlib.pyplot as plt
# numpy：数组与随机种子
import numpy as np
# pickle：读写本地 train/test 与模型缓存
import pickle
# Counter：统计特征名 / 品牌频次
from collections import Counter


In [ ]:
# ========== 传统机器学习相关导入 ==========

# pandas：特征表 DataFrame
import pandas as pd
# numpy：数值计算（与上格重复导入不影响逻辑）
import numpy as np
# 线性回归
from sklearn.linear_model import LinearRegression
# 回归指标：MSE、R²
from sklearn.metrics import mean_squared_error, r2_score
# 两种特征缩放器（后面 Word2Vec 实验会用）
from sklearn.preprocessing import StandardScaler, MinMaxScaler


In [ ]:
# ========== NLP 相关导入：词袋 + Word2Vec ==========

# CountVectorizer：Bag-of-Words 特征
from sklearn.feature_extraction.text import CountVectorizer
# Word2Vec：词向量模型（gensim）
from gensim.models import Word2Vec
# simple_preprocess：分词/小写等轻量预处理
from gensim.utils import simple_preprocess


In [ ]:
# ========== 更强的传统回归器导入 ==========

# LinearSVR：线性支持向量回归
from sklearn.svm import LinearSVR
# 随机森林回归
from sklearn.ensemble import RandomForestRegressor
# 梯度提升回归（本练习里当「xgb」变量使用）
from sklearn.ensemble import GradientBoostingRegressor
# polire.IDW：反距离加权（导入保留；本笔记本后续未必用到）
from polire import IDW


In [ ]:
# ========== 终端彩色输出常量（Tester 打印用） ==========

# ANSI 绿色
GREEN = "\033[92m"
# ANSI 黄色（映射为 orange）
YELLOW = "\033[93m"
# ANSI 红色
RED = "\033[91m"
# 重置颜色
RESET = "\033[0m"
# 颜色名 -> ANSI 码
COLOR_MAP = {"red":RED, "orange": YELLOW, "green": GREEN}


In [ ]:
# 让 matplotlib 图直接嵌在笔记本输出里（IPython magic）
%matplotlib inline


## 加载本地 pkl 数据

从当前目录读取预先准备好的 `train.pkl` / `test.pkl`（课程定价 Item 列表）。


In [ ]:
# ========== 反序列化训练集与测试集 ==========

# 二进制读 train.pkl
with open('./train.pkl', 'rb') as file:
    train = pickle.load(file)

# 二进制读 test.pkl
with open('./test.pkl', 'rb') as file:
    test = pickle.load(file)


In [ ]:
# ========== Tester：统一跑定价器、打点、算误差并画散点图 ==========

class Tester:

    def __init__(self, predictor, title=None, data=test, size=250):
        # 待测定价函数：Item -> 预测价格
        self.predictor = predictor
        # 默认用全局 test；也可传入别的列表
        self.data = data
        # 标题：默认从函数名生成
        self.title = title or predictor.__name__.replace("_", " ").title()
        # 评估条数（默认 250）
        self.size = size
        # 累积：预测、真值、绝对误差、SLE、颜色
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        # 绿：绝对误差 <40 或相对误差 <20%
        if error<40 or error/truth < 0.2:
            return "green"
        # 橙：绝对误差 <80 或相对误差 <40%
        elif error<80 or error/truth < 0.4:
            return "orange"
        # 红：其余
        else:
            return "red"
    
    def run_datapoint(self, i):
        # 取第 i 条测试样本
        datapoint = self.data[i]
        # 模型猜价
        guess = self.predictor(datapoint)
        # 真价
        truth = datapoint.price
        # 绝对误差
        error = abs(guess - truth)
        # 对数误差（用于 SLE / RMSLE）
        log_error = math.log(truth+1) - math.log(guess+1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        # 标题过长则截断，避免刷屏
        title = datapoint.title if len(datapoint.title) <= 40 else datapoint.title[:40]+"..."
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        # 彩色打印单条结果（英文格式串保持原样）
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:,.2f} Item: {title}{RESET}")

    def chart(self, title):
        max_error = max(self.errors)
        plt.figure(figsize=(12, 8))
        # 坐标轴上界：真价与预测的最大值
        max_val = max(max(self.truths), max(self.guesses))
        # y=x 参考线：落在线上表示估准了
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=2, alpha=0.6)
        # 散点：颜色编码误差档位
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel('Ground Truth')
        plt.ylabel('Model Estimate')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        # 平均绝对误差
        average_error = sum(self.errors) / self.size
        # 均方根对数误差
        rmsle = math.sqrt(sum(self.sles) / self.size)
        # 绿色命中条数
        hits = sum(1 for color in self.colors if color=="green")
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:,.2f} Hits={hits/self.size*100:.1f}%"
        self.chart(title)

    def run(self):
        self.error = 0
        # 顺序跑前 size 条
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, function):
        # 快捷入口：Tester.test(fn)
        cls(function).run()


In [ ]:
# ========== 基线：随机猜 1~999 ==========
def random_pricer(item):
    return random.randrange(1,1000)


In [ ]:
# ========== 固定种子后跑随机基线 ==========

# 设置随机种子，保证可复现
random.seed(42)

# 运行我们的测试运行程序
Tester.test(random_pricer)


In [ ]:
# ========== 基线：训练集均价常量模型 ==========
# 太有趣了！
# 我们可以做得更好——这是另一个相当简单的模型

# 抽出所有训练价格
training_prices = [item.price for item in train]
# 平均值
training_average = sum(training_prices) / len(training_prices)

# 无论输入什么商品，都返回同一个均价
def constant_pricer(item):
    return training_average


In [ ]:
# ========== 评估常量定价器 ==========
# 运行我们的常数预测器
Tester.test(constant_pricer)


In [ ]:
# ========== 把 details JSON 解析进 item.features ==========
# 在项目上创建一个新的“features”字段，并使用从详细信息字典解析的 json 填充它

for item in train:
    item.features = json.loads(item.details)
for item in test:
    item.features = json.loads(item.details)

# 看一个（下一格用 .keys()）


In [ ]:
# ========== 查看第一条训练样本有哪些特征键 ==========
train[0].features.keys()


In [ ]:
# ========== 统计训练集中最常见的特征字段名 ==========
# 查看训练集中 20 个最常见的特征（这里 most_common(40)）

feature_count = Counter()
for item in train:
    for f in item.features.keys():
        feature_count[f]+=1

# 返回频次最高的 40 个键
feature_count.most_common(40)


In [ ]:
# ========== 从 Item Weight 字符串解析重量（统一到磅） ==========
# 现在有一些糟糕的代码来提取物品重量
# 不要太担心这一点：剧透警告，它在训练中不会有多大用处！

def get_weight(item):
    # 从 features 取原始重量字符串
    weight_str = item.features.get('Item Weight')
    if weight_str:
        # 期望形如 "12 pounds" / "8 ounces" ...
        parts = weight_str.split(' ')
        # 数值部分
        amount = float(parts[0])
        # 单位部分小写，方便比较
        unit = parts[1].lower()
        if unit=="pounds":
            return amount
        elif unit=="ounces":
            # 16 盎司 = 1 磅
            return amount / 16
        elif unit=="grams":
            return amount / 453.592
        elif unit=="milligrams":
            return amount / 453592
        elif unit=="kilograms":
            return amount / 0.453592
        elif unit=="hundredths" and parts[2].lower()=="pounds":
            # hundredths pounds
            return amount / 100
        else:
            # 未知单位：打印原始串便于排查
            print(weight_str)
    return None


In [ ]:
# ========== 收集训练集中能解析出的重量，并去掉 None ==========
weights = [get_weight(t) for t in train]
weights = [w for w in weights if w]


In [ ]:
# ========== 算平均重量，供缺失值填充 ==========
average_weight = sum(weights)/len(weights)
average_weight


In [ ]:
# ========== 取重量：缺失则回落到平均重量 ==========
def get_weight_with_default(item):
    # 先走解析逻辑
    weight = get_weight(item)
    # None/0 都用训练集平均重量顶上
    return weight or average_weight


In [ ]:
# ========== 从 Best Sellers Rank 字典取平均排名 ==========
def get_rank(item):
    # 可能是 {类目名: 排名数字, ...}
    rank_dict = item.features.get("Best Sellers Rank")
    if rank_dict:
        # 多个类目排名：取算术平均做一个标量特征
        ranks = rank_dict.values()
        return sum(ranks)/len(ranks)
    # 没有该字段
    return None


In [ ]:
# ========== 统计可解析排名的平均值 ==========
# 训练集每条尝试取 rank
ranks = [get_rank(t) for t in train]
# 去掉 None
ranks = [r for r in ranks if r]
# 全局平均，供缺失填充
average_rank = sum(ranks)/len(ranks)
average_rank


In [ ]:
# ========== 取排名：缺失则用平均排名 ==========
def get_rank_with_default(item):
    # 能解析就用真值
    rank = get_rank(item)
    # 否则回落平均
    return rank or average_rank


In [ ]:
# ========== 文本长度特征：用 test_prompt() 的字符数 ==========
def get_text_length(item):
    return len(item.test_prompt())


In [ ]:
# ========== 调查品牌频次 ==========

# Counter：品牌 -> 出现次数
brands = Counter()
for t in train:
    # features 里可能没有 Brand
    brand = t.features.get("Brand")
    if brand:
        brands[brand]+=1

# 查看最常见的 40 个品牌
brands.most_common(40)


In [ ]:
# ========== 是否属于「头部电子品牌」的 0/1 特征 ==========
# 小写品牌白名单（匹配时会对 item 品牌 lower）
TOP_ELECTRONICS_BRANDS = ["hp", "dell", "lenovo", "samsung", "asus", "sony", "canon", "apple", "intel"]
def is_top_electronics_brand(item):
    brand = item.features.get("Brand")
    # 有品牌且落在白名单才为真
    return brand and brand.lower() in TOP_ELECTRONICS_BRANDS


In [ ]:
# ========== 汇总四维手工特征字典 ==========
def get_features(item):
    return {
        # 重量（缺失已 internally 填平均）
        "weight": get_weight_with_default(item),
        # 畅销排名（缺失填平均）
        "rank": get_rank_with_default(item),
        # 提示文本长度
        "text_length": get_text_length(item),
        # True/False -> 1/0，方便线性模型
        "is_top_electronics_brand": 1 if is_top_electronics_brand(item) else 0
    }


In [ ]:
# ========== 抽查：训练首条的特征向量 ==========
# 查看训练项目中的功能
get_features(train[0])


In [ ]:
# ========== Item 列表 -> 带 price 列的 DataFrame ==========
# 将我们的特征转换为 pandas 数据框的实用函数

def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

# 全量训练表
train_df = list_to_dataframe(train)
# 测试只取前 250，和 Tester 默认 size 对齐，加快实验
test_df = list_to_dataframe(test[:250])


In [ ]:
# ========== 传统线性回归：拟合 + 打印系数 + MSE/R² ==========
# 传统的线性回归！

np.random.seed(42)

# 单独的功能和目标
feature_columns = ['weight', 'rank', 'text_length', 'is_top_electronics_brand']

# 训练特征矩阵
X_train = train_df[feature_columns]
# 训练标签
y_train = train_df['price']
# 测试特征 / 标签（前 250）
X_test = test_df[feature_columns]
y_test = test_df['price']

# 训练线性回归
model = LinearRegression()
model.fit(X_train, y_train)

# 打印每个特征的系数，看方向与量级
for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# 预测测试集并评估
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")


In [ ]:
# ========== 包装成 Item -> 价格 的定价器 ==========
# 预测新商品价格的函数

def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]


In [ ]:
# ========== 用 Tester 评估手工特征线性回归 ==========
# 测试一下

Tester.test(linear_regression_pricer)


In [ ]:
# ========== 为词袋/向量模型准备语料与价格标签 ==========
# 对于接下来的几个型号，我们准备了文件和价格
# 请注意，我们使用文档的测试提示，否则我们将泄露答案！

# y：训练价格数组
prices = np.array([float(item.price) for item in train])
# X 文本：必须用 test_prompt()，避免把价格答案泄漏进特征
documents = [item.test_prompt() for item in train]


In [ ]:
# ========== 词袋 + 线性回归 ==========
# 将 CountVectorizer 用于词袋模型

np.random.seed(42)
# 最多 1000 词，去掉英文停用词
vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X = vectorizer.fit_transform(documents)
regressor = LinearRegression()
regressor.fit(X, prices)


In [ ]:
# ========== 词袋线性定价器，并预计算全 test 预测缓存 ==========
def bow_lr_pricer(item):
    x = vectorizer.transform([item.test_prompt()])
    # 价格下限截断为 0
    return max(regressor.predict(x)[0], 0)
# 预计算：后面 ensemble 要反复取同一预测
pred_lr = {}
for i in range(len(test)):
    pred_lr[test[i]] = bow_lr_pricer(test[i])


In [ ]:
# ========== 通过缓存字典接入 Tester ==========
# 测试一下
def get_pred_lr(item):
    return pred_lr[item]
Tester.test(get_pred_lr)


In [ ]:
# ========== 在同一词袋矩阵 X 上训练梯度提升（变量名 xgb） ==========
xgb = GradientBoostingRegressor()
xgb.fit(X, prices)


In [ ]:
# ========== 词袋 + 梯度提升定价器 ==========
def bow_xgb_pricer(item):
    # 与训练相同的 vectorizer，只 transform 不 fit
    x = vectorizer.transform([item.test_prompt()])
    # 负价格截到 0
    return max(xgb.predict(x)[0], 0)


In [ ]:
# ========== 预计算 xgb 预测并评估 ==========
# 测试
# 用 Item 对象当 key，缓存预测供 ensemble 复用
pred_xgb = {}
for i in range(len(test)):
    pred_xgb[test[i]] = bow_xgb_pricer(test[i])
# Tester 需要 Item -> 价格 的可调用对象
def get_pred_xgb(item):
    return pred_xgb[item]
Tester.test(get_pred_xgb)


In [ ]:
# ========== 加载 spaCy 英文小模型，供 NER 使用 ==========
import spacy 
nlp = spacy.load("en_core_web_sm")


In [ ]:
# ========== NER：抽出实体词，拼成「只含实体」的伪文档 ==========
def ner_doc(doc):
    # 换行变空格，避免奇怪分词
    d = nlp(doc.replace('\n',' '))
    ents = []
    for ent in d.ents:
        # 实体可能含多词：按空格拆开再收集
        ents.extend(ent.text.split(' '))
    # set 去重后再拼回字符串
    return ' '.join(list(set(ents)))
def ner_docs(docs):
    ret = []
    for i,doc in enumerate(docs):
        # 逐篇做 NER 过滤
        ret.append(ner_doc(doc))
        # 每 1000 条打印进度
        if i%1000 == 0:
            print(i, ret[-1])
    return ret


In [ ]:
# ========== 缓存 NER 结果到 docs.pkl（存在则直接读） ==========
if os.path.exists('docs.pkl'):
    docs2 = pickle.load(open('docs.pkl','rb'))
else:
    # 注意：此处调用的是 ner_doc(documents)——保持原逻辑不改
    docs2 = ner_doc(documents)
    pickle.dump(docs2, open('docs.pkl','wb'))


In [ ]:
# ========== 在 NER 文本上再做词袋 + 线性回归 ==========
np.random.seed(42)
# 第二套词袋：吃的是实体词伪文档
vectorizer2 = CountVectorizer(max_features=1000, stop_words='english')
X2 = vectorizer2.fit_transform(docs2)
regressor2 = LinearRegression()
# 标签仍是原始 prices
regressor2.fit(X2, prices)


In [ ]:
# ========== NER 词袋定价器：先 ner_doc 再 transform ==========
def ner_pricer(item):
    # 推理路径必须与训练一致：先抽实体再向量化
    x = vectorizer2.transform([ner_doc(item.test_prompt())])
    return max(regressor2.predict(x)[0], 0)
# 测试：先缓存全量预测
pred_ner = {}
for i in range(len(test)):
    pred_ner[test[i]] = ner_pricer(test[i])
def get_pred_ner(item):
    return pred_ner[item]
Tester.test(get_pred_ner)


In [ ]:
# ========== 训练 Word2Vec 词向量 ==========
# 令人惊叹的 word2vec 模型，在 gensim NLP 库中实现

np.random.seed(42)

# 预处理文档：变成 token 列表
processed_docs = [simple_preprocess(doc) for doc in documents]

# 训练 Word2Vec：400 维，窗口 5，保留低频词，8 线程
w2v_model = Word2Vec(sentences=processed_docs, vector_size=400, window=5, min_count=1, workers=8)


In [ ]:
# ========== 文档向量 = 词向量均值（简单但信息有损） ==========
# 对整个文档中的向量进行平均的这一步骤是我们方法的一个弱点

def document_vector(doc):
    doc_words = simple_preprocess(doc)
    # 只收集词表里有的词向量
    word_vectors = [w2v_model.wv[word] for word in doc_words if word in w2v_model.wv]
    # 没空词则退回零向量，维度与模型一致
    return np.mean(word_vectors, axis=0) if word_vectors else np.zeros(w2v_model.vector_size)

# 创建特征矩阵：每行一篇文档
X_w2v = np.array([document_vector(doc) for doc in documents])


In [ ]:
# ========== Word2Vec 特征 + 线性回归 ==========
# 在 word2vec 上运行线性回归

word2vec_lr_regressor = LinearRegression()
word2vec_lr_regressor.fit(X_w2v, prices)


In [ ]:
# ========== Word2Vec 线性定价器 ==========
def word2vec_lr_pricer(item):
    # 与训练相同：用 test_prompt 文本
    doc = item.test_prompt()
    # 文档向量 = 词向量均值
    doc_vector = document_vector(doc)
    # 预测并截负
    return max(0, word2vec_lr_regressor.predict([doc_vector])[0])


In [ ]:
# ========== 缓存 w2v 线性预测并评估 ==========
pred_lr_w2v = {}
for i in range(len(test)):
    # 逐条推理写入缓存
    pred_lr_w2v[test[i]] = word2vec_lr_pricer(test[i])
def get_pred_lr_w2v(item):
    return pred_lr_w2v[item]
Tester.test(get_pred_lr_w2v)


In [ ]:
# 专家荟萃（ensemble）：下面两格试不同聚合方式


In [ ]:
# ========== 专家组合：在三模型中选「最接近的一对」取平均 ==========
def get_pred_best2_mean(item):
    # 三个已缓存的专家预测
    v1 = pred_lr[item]
    v2 = pred_lr_w2v[item]
    v3 = pred_xgb[item]
    # 三对两两绝对差
    d1 = abs(v1-v2)
    d2 = abs(v2-v3)
    d3 = abs(v3-v1)
    # 差值最小的那一对更「意见一致」，取它们的均值
    if d1 <= min(d2,d3):
        v = (v1+v2)/2
    elif d2 <= min(d1,d3):
        v = (v2+v3)/2
    else:
        v = (v1+v3)/2
    return v
Tester.test(get_pred_best2_mean)


In [ ]:
# ========== 专家组合：三模型简单平均 ==========
def get_pred_mean(item):
    # 取三个缓存预测
    v1 = pred_lr[item]
    v2 = pred_lr_w2v[item]
    v3 = pred_xgb[item]
    # 算术平均，压制单模型极端值
    return (v1+v2+v3)/3
Tester.test(get_pred_mean)


In [ ]:
# 应用 MinMax 和标准缩放器（下一格实际选用 MinMaxScaler）


In [ ]:
# ========== 用 MinMaxScaler 拟合并变换 Word2Vec 特征 ==========
# [MinMaxScaler, StandardScaler][0] 等价于选用 MinMaxScaler
scalar = [MinMaxScaler, StandardScaler][0]().fit(X_w2v)
X_w2v_scaled = scalar.transform(X_w2v)


In [ ]:
# ========== 在缩放后的 Word2Vec 特征上拟合线性回归 ==========
word2vec_lr_reg_scaled = LinearRegression().fit(X_w2v_scaled, prices)


In [ ]:
# ========== 缩放版 w2v 线性定价器（变量名保持原样：stdscalar） ==========
def word2vec_lr_pricer_scaled(item):
    doc = item.test_prompt()
    # 先得到原始文档向量
    doc_vector = document_vector(doc)
    # 注意：此处原代码使用 stdscalar；逻辑保持不改
    doc_vector_scaled = stdscalar.transform([doc_vector])
    # 用缩放空间上训好的线性模型预测
    return max(0, word2vec_lr_reg_scaled.predict([doc_vector_scaled[0]])[0])

Tester.test(word2vec_lr_pricer_scaled)


In [ ]:
# ========== Word2Vec 特征 + 梯度提升 ==========
# 在 word2vec 上运行 XGB

word2vec_xgb_regressor = GradientBoostingRegressor()
word2vec_xgb_regressor.fit(X_w2v, prices)


In [ ]:
# ========== Word2Vec + 梯度提升定价器 ==========
def word2vec_xgb_pricer(item):
    doc = item.test_prompt()
    # 同一套 document_vector
    doc_vector = document_vector(doc)
    return max(0, word2vec_xgb_regressor.predict([doc_vector])[0])


In [ ]:
# ========== 评估 w2v + 梯度提升 ==========
Tester.test(word2vec_xgb_pricer)


In [ ]:
# ========== 支持向量回归（LinearSVR）跑在 Word2Vec 特征上 ==========
# 支持向量机

np.random.seed(42)
svr_regressor = LinearSVR()

svr_regressor.fit(X_w2v, prices)


In [ ]:
# ========== SVR 定价器 ==========
def svr_pricer(item):
    # 与训练时一样固定种子（LinearSVR 内部若有随机性）
    np.random.seed(42)
    doc = item.test_prompt()
    doc_vector = document_vector(doc)
    # float() 确保返回 Python 标量；再截负
    return max(float(svr_regressor.predict([doc_vector])[0]),0)


In [ ]:
# ========== 评估 SVR ==========
Tester.test(svr_pricer)


In [ ]:
# ========== 随机森林：有缓存 pkl 就加载，否则现场训练 ==========
# 以及强大的随机森林回归
# 本地缓存文件名
mfile = 'random_forest_model.pkl'
if os.path.exists(mfile):
    # 命中缓存：跳过漫长 fit
    rf_model = pickle.load(open(mfile,'rb'))
else:
    # 100 棵树，8 线程；特征是 Word2Vec 文档向量
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=8)
    rf_model.fit(X_w2v, prices)


In [ ]:
# ========== 随机森林定价器（Word2Vec 文档向量） ==========
def random_forest_pricer(item):
    doc = item.test_prompt()
    doc_vector = document_vector(doc)
    return max(0, rf_model.predict([doc_vector])[0])


In [ ]:
# ========== 直接评估随机森林（边测边推理） ==========
Tester.test(random_forest_pricer)


In [ ]:
# ========== 预计算 RF 预测后再评估（与其它 ensemble 缓存风格一致） ==========
pred_rf = {}
for i in range(len(test)):
    pred_rf[test[i]] = random_forest_pricer(test[i])
def get_pred_rf(item):
    return pred_rf[item]
Tester.test(get_pred_rf)
